In [15]:
import joblib
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [16]:

X_train_processed = joblib.load('/content/drive/MyDrive/Colab Notebooks/AIRBNB/X_train_processed.pkl')
X_test_processed = joblib.load('/content/drive/MyDrive/Colab Notebooks/AIRBNB/X_test_processed.pkl')
y_train = joblib.load('/content/drive/MyDrive/Colab Notebooks/AIRBNB/y_train.pkl')
y_test = joblib.load('/content/drive/MyDrive/Colab Notebooks/AIRBNB/y_test.pkl')

print(f"Loaded X_train shape: {X_train_processed.shape}")
print(f"Loaded X_test shape: {X_test_processed.shape}")

Loaded X_train shape: (39084, 90)
Loaded X_test shape: (9771, 90)


In [17]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Lasso Regression": Lasso(alpha=0.01, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
}


In [18]:
def evaluate_model(model_name, y_true_log, y_pred_log):
    y_true_actual = np.expm1(y_true_log)
    y_pred_actual = np.expm1(y_pred_log)

    rmse = np.sqrt(mean_squared_error(y_true_actual, y_pred_actual))
    mae = mean_absolute_error(y_true_actual, y_pred_actual)
    r2 = r2_score(y_true_log, y_pred_log)

    print(f"--- {model_name} ---")
    print(f"R-squared (Log Space): {r2:.4f}")
    print(f"RMSE (Actual Dollars): ${rmse:.2f}")
    print(f"MAE (Actual Dollars):  ${mae:.2f}\n")

In [19]:
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    evaluate_model(name, y_test, y_pred)

--- Linear Regression ---
R-squared (Log Space): 0.5796
RMSE (Actual Dollars): $204.58
MAE (Actual Dollars):  $56.12

--- Ridge Regression ---
R-squared (Log Space): 0.5796
RMSE (Actual Dollars): $204.58
MAE (Actual Dollars):  $56.12

--- Lasso Regression ---
R-squared (Log Space): 0.5285
RMSE (Actual Dollars): $208.16
MAE (Actual Dollars):  $58.94

--- Random Forest ---
R-squared (Log Space): 0.6067
RMSE (Actual Dollars): $201.77
MAE (Actual Dollars):  $54.66

--- Gradient Boosting ---
R-squared (Log Space): 0.6200
RMSE (Actual Dollars): $200.70
MAE (Actual Dollars):  $53.91

--- XGBoost ---
R-squared (Log Space): 0.6243
RMSE (Actual Dollars): $200.37
MAE (Actual Dollars):  $53.48

--- LightGBM ---
R-squared (Log Space): 0.6235
RMSE (Actual Dollars): $200.74
MAE (Actual Dollars):  $53.73



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [20]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

param_grid = {
    'n_estimators': [200, 350],
    'learning_rate': [0.05, 0.1],
    'max_depth': [6, 8],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=3,               # 3-fold cross-validation
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_processed, y_train)

print(f"Best Hyperparameters Found: {grid_search.best_params_}")
best_xgb_model = grid_search.best_estimator_



Fitting 3 folds for each of 32 candidates, totalling 96 fits
Best Hyperparameters Found: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 200, 'subsample': 0.8}


In [21]:
# Final Evaluation on the Test Set
print("\nEvaluating Final Grid-Tuned Model:")
y_pred_tuned = best_xgb_model.predict(X_test_processed)
evaluate_model("Grid-Tuned XGBoost (Final)", y_test, y_pred_tuned)


Evaluating Final Grid-Tuned Model:
--- Grid-Tuned XGBoost (Final) ---
R-squared (Log Space): 0.6298
RMSE (Actual Dollars): $199.60
MAE (Actual Dollars):  $53.21



In [22]:
# Save the final optimized model to disk
joblib.dump(best_xgb_model, 'best_airbnb_model_grid.pkl')
print("Final grid-tuned model successfully saved as 'best_airbnb_model_grid.pkl'!")

Final grid-tuned model successfully saved as 'best_airbnb_model_grid.pkl'!
